In [3]:
import pandas as pd 
import numpy as np
import statsmodels.api as sm
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import table
from reportlab.lib.pagesizes import letter, landscape
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Spacer
from reportlab.lib import colors

Using Daily data resample weekly data

In [5]:
import pandas as pd  
import numpy as np
import statsmodels.api as sm
import glob
import os



###############################################################################
# 辅助函数：根据 p 值格式化系数（附加显著性星号）
###############################################################################
def format_coef(coef, p):
    if np.isnan(coef) or np.isnan(p):
        return f"{coef:.4f}"
    star = ""
    if p < 0.01:
        star = "***"
    elif p < 0.05:
        star = "**"
    elif p < 0.1:
        star = "*"
    return f"{coef:.4f}{star}"

###############################################################################
# 0. 日频数据聚合为周频数据（使用 ISO 8601 周规则）
###############################################################################
def aggregate_daily_to_weekly(df):
    # 1) 时间转换 & 筛选日期
    df = df.rename(columns={"time_open": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    # 如有需要可筛选时间范围，例如：
    df = df.query('"2014-01-01" <= Date <= "2020-07-31"')
    
    # 2) 使用 ISO 8601 周规则：获取 ISO 年和 ISO 周
    iso = df["Date"].dt.isocalendar()
    df["iso_year"] = iso.year
    df["iso_week"] = iso.week

    # 3) 数据清洗：剔除缺失及异常值；同时对市值筛选（大于 1,000,000）
    df = df.dropna()
    df = df[(df["volume"] > 0) & (df["close"] > 0)]
    df = df.dropna(subset=["market_cap"])
    df = df[df["market_cap"] > 1_000_000]
    
    # 4) 计算每日价格成交额及其对数
    df["price_vol"] = df["close"] * df["volume"]
    df["log_price_vol"] = np.log(df["price_vol"])
    
    # 5) 按币种、ISO年和ISO周聚合（聚合规则与论文一致）
    agg_dict = {
        "Date": "max",          # 每只币当周最后交易日
        "close": "last",        # 本周最后一天收盘价
        "high": "max",          # 本周最高价
        "market_cap": "last",   # 本周最后一天市值
        "volume": "mean",       # 本周平均交易量
        "price_vol": "mean",    # 本周平均成交额
        "log_price_vol": "std"  # 本周对数成交额样本标准差（ddof=1）
    }
    weekly = df.groupby(["id", "iso_year", "iso_week"], as_index=False).agg(agg_dict)
    
    # 6) 重命名聚合列 & 生成对数特征
    weekly.rename(columns={
        "Date": "Date_max",
        "close": "close_last",
        "high": "high_max",
        "market_cap": "market_cap_last",
        "volume": "volume_mean",
        "price_vol": "price_vol_mean",
        "log_price_vol": "log_price_vol_std"
    }, inplace=True)
    weekly["MCAP"] = np.log(weekly["market_cap_last"])
    weekly["PRC"] = np.log(weekly["close_last"])
    weekly["MAXDPRC"] = np.log(weekly["high_max"])
    weekly["PRCVOL"] = np.log(weekly["price_vol_mean"])
    weekly.rename(columns={"log_price_vol_std": "STDPRCVOL"}, inplace=True)
    
    # 7) 计算周度对数收益率（按每只币的收盘价差分）
    weekly.sort_values(["id", "Date_max"], inplace=True)
    # weekly["rtn"] = weekly.groupby("id")["close_last"].transform(lambda x: np.log(x).diff())
    weekly["rtn"] = weekly.groupby("id")["close_last"].transform(lambda x: x.pct_change())

    
    # 8) 删除因差分产生的缺失值
    weekly.dropna(subset=["rtn"], inplace=True)
    
    # 9) 统一每周日期：使用 ISO 年、ISO 周及星期日作为代表日期
    weekly["Date"] = weekly.apply(lambda row: pd.Timestamp.fromisocalendar(int(row["iso_year"]), int(row["iso_week"]), 7), axis=1)
    print(weekly)
    return weekly

###############################################################################
# 1. 读取并聚合加密货币日频数据（文件名格式：crypto_full_*_daily_data.csv）
###############################################################################
def load_crypto_data(folder_path):
    file_paths = sorted(glob.glob(os.path.join(folder_path, "crypto_full_*_daily_data.csv")))
    daily_df = pd.concat((pd.read_csv(fp) for fp in file_paths), ignore_index=True)
    weekly_df = aggregate_daily_to_weekly(daily_df)
    return weekly_df

def preprocess_crypto_data(df):
    # 排序 & 删除因收益率差分产生的缺失值
    df = df.sort_values(["id", "Date"]).reset_index(drop=True)
    df = df.dropna(subset=["rtn"])
    
    # 生成对数特征（如未生成）
    if "MCAP" not in df.columns:
        df["MCAP"] = np.log(df["market_cap_last"])
    if "PRC" not in df.columns:
        df["PRC"] = np.log(df["close_last"])
    if "MAXDPRC" not in df.columns:
        df["MAXDPRC"] = np.log(df["high_max"]) if "high_max" in df.columns else np.log(df["close_last"])
    
    # 构造滞后收益率特征
    df["r_1_0"] = df.groupby("id")["rtn"].shift(1)
    df["r_2_0"] = df.groupby("id")["rtn"].shift(2)
    df["r_3_0"] = df.groupby("id")["rtn"].shift(3)
    df["r_4_0"] = df.groupby("id")["rtn"].shift(4)
    df["r_4_1"] = df[["r_1_0", "r_2_0", "r_3_0", "r_4_0"]].mean(axis=1)
    
    df["volume"] = df["volume_mean"]
    return df

###############################################################################
# 2. 读取国债日频数据，转换为周频数据，并计算超额收益
###############################################################################
def merge_riskfree_and_compute_excess(df, rf_path):
    rf_data = pd.read_csv(rf_path)
    rf_data["Date"] = pd.to_datetime(rf_data["Date"])
    # 假设 "1 Mo" 为日频国债收益率（百分比），转换为小数
    rf_data["Rf"] = rf_data["1 Mo"] / 100
    # 按周均值转换为周频数据（默认W频率，标签为周日，与我们的聚合一致）
    rf_data = rf_data.set_index("Date").resample("W").mean().reset_index()
    df = pd.merge_asof(df.sort_values("Date"),
                       rf_data[["Date", "Rf"]].sort_values("Date"),
                       on="Date", direction="backward")
    df["excess_return"] = df["rtn"] - df["Rf"]
    return df

###############################################################################
# 3. 构造三大因子：CMKT, CSMB, CMOM（市值加权）
###############################################################################
def weighted_mean(g, value_col="excess_return", weight_col="market_cap_last"):
    return (g[value_col] * g[weight_col]).sum() / g[weight_col].sum()

def build_3factors(df):
    # CMKT: 市值加权超额收益
    CMKT = df.groupby("Date").apply(
        lambda g: np.sum(g["excess_return"] * g["market_cap_last"]) / np.sum(g["market_cap_last"])
    ).rename("CMKT")
    
    # CSMB: 每个日期内基于当周数据按市值分位数划分小盘、中盘、大盘，计算小盘与大盘收益差
    def assign_size(g):
        q30 = g["market_cap_last"].quantile(0.3)
        q70 = g["market_cap_last"].quantile(0.7)
        g = g.copy()
        g["Size_Group"] = "Big"
        g.loc[g["market_cap_last"] <= q30, "Size_Group"] = "Small"
        g.loc[(g["market_cap_last"] > q30) & (g["market_cap_last"] <= q70), "Size_Group"] = "Middle"
        return g
    df_size = df.groupby("Date").apply(assign_size).reset_index(drop=True)
    
    small_ret = df_size[df_size["Size_Group"] == "Small"].groupby("Date").apply(
        lambda g: weighted_mean(g, "excess_return", "market_cap_last")
    )
    big_ret = df_size[df_size["Size_Group"] == "Big"].groupby("Date").apply(
        lambda g: weighted_mean(g, "excess_return", "market_cap_last")
    )
    CSMB = (small_ret - big_ret).rename("CSMB")
    
    # CMOM: 动量因子基于过去3周原始收益（rtn）
    # 每个日期内先按市值分位数，仅保留底30%与顶30%的币种
    def assign_size_mom(g):
        q30 = g["market_cap_last"].quantile(0.3)
        q70 = g["market_cap_last"].quantile(0.7)
        g = g[(g["market_cap_last"] <= q30) | (g["market_cap_last"] > q70)].copy()
        g["Size_Group"] = np.where(g["market_cap_last"] <= q30, "Small", "Big")
        return g
    df_mom = df.groupby("Date").apply(assign_size_mom).reset_index(drop=True)
    
    # 计算过去3周动量（复合收益率），使用原始收益 (rtn)
    df_mom["momentum"] = df_mom.groupby("id")["rtn"].transform(
        lambda x: (1 + x).rolling(window=3, min_periods=3).apply(np.prod, raw=True) - 1
    )
    df_mom = df_mom.dropna(subset=["momentum"])
    
    # 在每个日期和市值组内，根据动量排序分为三组：底部30%（Low）、中间40%（Mid）、顶部30%（High）
    def assign_mom(sub):
        sub = sub.sort_values("momentum").reset_index(drop=True)
        n = len(sub)
        if n < 3:
            sub["MomGroup"] = np.nan
            return sub
        c1 = int(np.floor(n * 0.3))
        c2 = int(np.floor(n * 0.7))
        sub.loc[:c1-1, "MomGroup"] = "Low"
        sub.loc[c1:c2-1, "MomGroup"] = "Mid"
        sub.loc[c2:, "MomGroup"] = "High"
        return sub
    df_mom = df_mom.groupby(["Date", "Size_Group"], group_keys=False).apply(assign_mom)
    
    def compute_cmom(g):
        sH = weighted_mean(g[(g["Size_Group"] == "Small") & (g["MomGroup"] == "High")],
                             "rtn", "market_cap_last")
        bH = weighted_mean(g[(g["Size_Group"] == "Big") & (g["MomGroup"] == "High")],
                             "rtn", "market_cap_last")
        sL = weighted_mean(g[(g["Size_Group"] == "Small") & (g["MomGroup"] == "Low")],
                             "rtn", "market_cap_last")
        bL = weighted_mean(g[(g["Size_Group"] == "Big") & (g["MomGroup"] == "Low")],
                             "rtn", "market_cap_last")
        if pd.isnull(sH) or pd.isnull(bH) or pd.isnull(sL) or pd.isnull(bL):
            return np.nan
        return 0.5 * (sH + bH) - 0.5 * (sL + bL)
    
    CMOM = df_mom.groupby("Date").apply(compute_cmom).rename("CMOM")
    
    factors = pd.concat([CMKT, CSMB, CMOM], axis=1)
    return factors

###############################################################################
# 4. 构造10个多空策略（Q5 - Q1），采用市值加权计算多空组合收益差
###############################################################################
def build_long_short_strategies(df):
    feats_10 = ["MCAP", "PRC", "MAXDPRC", "r_1_0", "r_2_0", "r_3_0", "r_4_0", "r_4_1", "PRCVOL", "STDPRCVOL"]
    ls_list = []
    for fac in feats_10:
        ls_diff = []
        for date_i, sub in df.groupby("Date"):
            sub_valid = sub.dropna(subset=[fac, "excess_return", "market_cap_last"])
            if len(sub_valid) < 5:
                continue
            sub_valid = sub_valid.sort_values(fac)
            q = len(sub_valid) // 5
            if q < 1:
                continue
            low_ret = weighted_mean(sub_valid.iloc[:q], "excess_return", "market_cap_last")
            high_ret = weighted_mean(sub_valid.iloc[-q:], "excess_return", "market_cap_last")
            ls_diff.append((date_i, high_ret - low_ret))
        tmp = pd.DataFrame(ls_diff, columns=["Date", fac + "_LS"]).set_index("Date")
        ls_list.append(tmp)
    ls_merged = pd.concat(ls_list, axis=1).dropna()
    return ls_merged

###############################################################################
# 5. 多因子回归（生成论文 Table IX 风格的结果）
###############################################################################
def run_ols(data, y_col, x_cols):
    X = sm.add_constant(data[x_cols])
    y = data[y_col]
    model = sm.OLS(y, X, missing='drop').fit()
    return model

def run_regression_table(ls_merged, factors):
    common_idx = ls_merged.index.intersection(factors.index)
    ls_merged = ls_merged.loc[common_idx]
    factors   = factors.loc[common_idx]
    data_all = ls_merged.join(factors, how="inner")
    
    feats_10 = ["MCAP", "PRC", "MAXDPRC", "r_1_0", "r_2_0", "r_3_0", "r_4_0", "r_4_1", "PRCVOL", "STDPRCVOL"]
    results_rows = []
    for fac in feats_10:
        y_var = fac + "_LS"
        model_1f = run_ols(data_all, y_var, ["CMKT", "CSMB"])
        model_2f = run_ols(data_all, y_var, ["CMKT", "CMOM"])
        model_3f = run_ols(data_all, y_var, ["CMKT", "CSMB", "CMOM"])
        
        for idx, m in enumerate([model_1f, model_2f, model_3f], start=1):
            coefs = m.params
            tvals = m.tvalues
            pvals = m.pvalues
            row = {
                "Strategy": fac,
                "(Spec)": idx,
                "Cons": format_coef(coefs.get("const", np.nan), pvals.get("const", np.nan)),
                "t(Cons)": tvals.get("const", np.nan),
                "CMKT": format_coef(coefs.get("CMKT", np.nan), pvals.get("CMKT", np.nan)),
                "t(CMKT)": tvals.get("CMKT", np.nan),
                "CSMB": format_coef(coefs.get("CSMB", np.nan), pvals.get("CSMB", np.nan)),
                "t(CSMB)": tvals.get("CSMB", np.nan),
                "CMOM": format_coef(coefs.get("CMOM", np.nan), pvals.get("CMOM", np.nan)),
                "t(CMOM)": tvals.get("CMOM", np.nan),
                "R^2": m.rsquared,
                "m.a.e.": abs(m.resid).mean(),
                "Avg R^2": m.rsquared_adj,
                "N": m.nobs
            }
            results_rows.append(row)
    results_df = pd.DataFrame(results_rows).sort_values(["Strategy", "(Spec)"]).reset_index(drop=True)
    return results_df

###############################################################################
# 6. 单因子回归（仅针对 CMKT）
###############################################################################
def single_factor_regression_table(ls_merged, cmkt_series):
    common_idx = ls_merged.index.intersection(cmkt_series.index)
    ls_merged  = ls_merged.loc[common_idx]
    cmkt_series = cmkt_series.loc[common_idx]
    data_all = ls_merged.copy()
    data_all["CMKT"] = cmkt_series

    results_rows = []
    for strategy_col in ls_merged.columns:
        Y = data_all[strategy_col]
        X = sm.add_constant(data_all["CMKT"])
        model = sm.OLS(Y, X, missing='drop').fit()
        row = {
            "Strategy": strategy_col,
            "alpha": format_coef(model.params.get("const", np.nan), model.pvalues.get("const", np.nan)),
            "t(alpha)": model.tvalues.get("const", np.nan),
            "beta_CMKT": format_coef(model.params.get("CMKT", np.nan), model.pvalues.get("CMKT", np.nan)),
            "t(beta_CMKT)": model.tvalues.get("CMKT", np.nan),
            "R^2": model.rsquared,
            "m.a.e.": abs(model.resid).mean(),
            "Avg R^2": model.rsquared_adj,
            "N": model.nobs
        }
        results_rows.append(row)
    results_df = pd.DataFrame(results_rows).sort_values("Strategy").reset_index(drop=True)
    return results_df


from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib import colors

def df_to_pdf(df, pdf_filename):
    doc = SimpleDocTemplate(
        pdf_filename,
        pagesize=landscape(letter),
        leftMargin=0, rightMargin=0, topMargin=0, bottomMargin=0
    )
    df = df.applymap(lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else x)
    data = [df.columns.tolist()] + df.values.tolist()
    table = Table(data)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 0), (-1, -1), 10),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 5),
        ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
    ]))
    doc.build([table])
    print(f"✅ PDF 文件已生成: {pdf_filename}")


###############################################################################
# 7. 主函数：整合所有流程，复现论文结果
###############################################################################
def main():
    # 1) 读取日频加密货币数据并聚合为周频数据
    folder_path = r"D:\桌面\绿色货币\数据总库\R数据抓取加密货币数据"
    df = load_crypto_data(folder_path)
    
    # 2) 数据预处理及特征构造
    df = preprocess_crypto_data(df)
    
    # 3) 合并国债数据，并计算超额收益（excess_return = rtn - Rf）
    rf_path = r"D:\桌面\绿色货币\数据总库\宏观经济数据\美国国债\us_treasury_daliy_1M.csv"
    df = merge_riskfree_and_compute_excess(df, rf_path)
    
    # 4) 构造三大因子（CMKT, CSMB, CMOM）
    factors = build_3factors(df)
    print("\n========== Factors (CMKT, CSMB, CMOM) ==========")
    print(factors.head())
    print(factors.describe())
    factors.to_csv('factors.csv', index=True)
    
    # 5) 构造 10 个多空策略（Q5 - Q1）
    ls_merged = build_long_short_strategies(df)
    print("\n========== 10 Long-Short Strategies (Q5 - Q1) ==========")
    print(ls_merged.head())
    print(ls_merged.describe())

    # 6) 多因子回归（生成论文 Table IX 风格结果）
    res_df = run_regression_table(ls_merged, factors)
    print("\n===================== Table IX (Final) =====================")
    print(res_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    res_df.to_csv("Daytoweek_20142020_3_factor_regression_results.csv", index=False, float_format="%.4f")
    df_to_pdf(res_df, "Daytoweek_20142020_3_factor_regression_results.pdf")
    print("回归结果表尺寸：", res_df.shape) 
    
    # 7) 单因子回归（仅针对 CMKT）
    single_res_df = single_factor_regression_table(ls_merged, factors["CMKT"])
    print("\n===================== Single-Factor Regression (CMKT) =====================")
    print(single_res_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    single_res_df.to_csv("Daytoweek_20142020_CMKT_factor_regression_results.csv", index=False, float_format="%.4f")
    df_to_pdf(single_res_df, "Daytoweek_20142020_CMKT_factor_regression_results.pdf")
    
    
if __name__ == "__main__":
    main()


C:\Users\why97\AppData\Local\Temp\ipykernel_16968\1773140524.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["iso_year"] = iso.year
C:\Users\why97\AppData\Local\Temp\ipykernel_16968\1773140524.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["iso_week"] = iso.week
C:\Users\why97\AppData\Local\Temp\ipykernel_16968\1773140524.py:145: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns wi


========== Factors (CMKT, CSMB, CMOM) ==========
                CMKT      CSMB      CMOM
Date                                    
2014-01-12 -0.083272 -0.081700       NaN
2014-01-19  0.010203 -0.161190       NaN
2014-01-26  0.016959 -0.076006  0.109917
2014-02-02 -0.070715 -0.081414  0.194485
2014-02-09 -0.191152 -0.067176  0.127368
             CMKT        CSMB        CMOM
count  343.000000  343.000000  340.000000
mean     0.002622   -0.037736    0.183221
std      0.110839    0.090915    0.132643
min     -0.421741   -0.373960   -0.485688
25%     -0.045398   -0.078648    0.109071
50%      0.002894   -0.037508    0.161329
75%      0.056062    0.001534    0.238841
max      0.383272    0.467069    0.810224

========== 10 Long-Short Strategies (Q5 - Q1) ==========
             MCAP_LS    PRC_LS  MAXDPRC_LS  r_1_0_LS  r_2_0_LS  r_3_0_LS  \
Date                                                                       
2014-02-09  0.026263 -0.178599   -0.178599 -0.230768  0.181136  0.346638   

C:\Users\why97\AppData\Local\Temp\ipykernel_16968\1773140524.py:329: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else x)
C:\Users\why97\AppData\Local\Temp\ipykernel_16968\1773140524.py:329: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else x)
